# Path 3B — The Spark World: Managed Spark MLlib Re-Ranker + `SparkMonitor` + GEAP (fka Vertex AI) Experiments (`@skill:gcp-spark`, `@skill:ml-best-practices`)
- **Engine**: Managed Service for Apache Spark (Dataproc Serverless Runtime `2.3` — Spark 3.5 + Iceberg `1.5.0`, matching `lakehouse-demo`) via `clarity-lrc-spark-template`.
- **Observability & Tracking**:
  - `SparkMonitor` displays live Spark stages, executors, and task progress inline below the cell.
  - **GEAP (fka Vertex AI) Experiments** (`google.cloud.aiplatform`) logs hyperparameters (`max_iter`, `max_depth`, debiasing flag) and chronological holdout `ROC AUC` metrics for governance and experiment comparison.
- **ML Best Practices**:
  1. `df.printSchema()` before referencing columns (`@skill:gcp-spark`).
  2. Chronological 80/20 split on `event_timestamp` (`@skill:ml-best-practices`).
  3. Counterfactual Rank-1 position-bias debiasing (`withColumn("displayed_rank_position", lit(1))`).
  4. `vector_to_array(col("probability")).getItem(1)` conversion before BigQuery / BigLake Iceberg export (`@skill:gcp-spark/resources/ml_tasks.md`).

In [ ]:
import os
from typing import Any
import google.auth
import google.auth.transport.requests
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

PROJECT_ID = "your-gcp-project-id"
LOCATION = "us-central1"
SESSION_TEMPLATE = "clarity-lrc-spark-template"
WAREHOUSE_BUCKET = f"gs://{PROJECT_ID}-iceberg-lakehouse"

FEATURE_NUMERIC_COLS = [
    "query_length_tokens",
    "displayed_rank_position",
    "lexical_bm25_score",
    "semantic_cosine_similarity",
    "historical_item_ctr",
    "price_usd",
    "average_rating",
]

# Refresh ADC credentials and construct Bearer token for Iceberg 1.5.0 REST Catalogs
_creds, _ = google.auth.default()
_creds.refresh(google.auth.transport.requests.Request())
_auth_header = "Bearer " + _creds.token

if os.path.exists("/usr/lib/spark"):
    # Connected directly to Remote Dataproc Serverless Jupyter Kernel (Runtime 2.3 via VS Code DAK Kernel Picker)
    spark = (
        SparkSession.builder.appName("clarity-spark-mllib-geap-notebook")
        .config("spark.sql.catalog.lrc", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lrc.type", "rest")
        .config("spark.sql.catalog.lrc.uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog")
        .config("spark.sql.catalog.lrc.warehouse", WAREHOUSE_BUCKET)
        .config("spark.sql.catalog.lrc.header.x-goog-user-project", PROJECT_ID)
        .config("spark.sql.catalog.lrc.header.X-Iceberg-Access-Delegation", "vended-credentials")
        .config("spark.sql.catalog.lrc.header.Authorization", _auth_header)
        .config("spark.sql.catalog.lrc.io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO")
        .getOrCreate()
    )
    spark.conf.set("spark.sql.catalog.lrc.header.Authorization", _auth_header)
else:
    # Connected to Local Python Kernel -> Use DataprocSparkSession with Runtime 2.3 Session Template
    from google.cloud.dataproc_spark_connect import DataprocSparkSession
    from google.cloud.dataproc_v1 import Session

    _session = Session()
    _session.session_template = f"projects/{PROJECT_ID}/locations/{LOCATION}/sessionTemplates/{SESSION_TEMPLATE}"
    for _cat in ("lrc", "biglakefederated"):
        _session.runtime_config.properties[f"spark.sql.catalog.{_cat}.header.Authorization"] = _auth_header

    spark = (
        DataprocSparkSession.builder.appName("clarity-spark-mllib-geap-notebook")
        .dataprocSessionConfig(_session)
        .getOrCreate()
    )


def train_and_score_spark_mllib(
    impressions_df: DataFrame,
    max_iter: int = 25,
    max_depth: int = 5,
) -> tuple[DataFrame, float]:
    """Train distributed GBTClassifier with chronological split and return debiased predictions."""
    impressions_df.printSchema()

    prepared_df = impressions_df.withColumn("event_ts", F.to_timestamp("event_timestamp")).withColumn(
        "purchased_label", F.col("purchased").cast("double")
    )

    split_epoch = prepared_df.select(
        F.percentile_approx(F.unix_timestamp("event_ts"), 0.80).alias("cutoff")
    ).collect()[0]["cutoff"]

    train_df = prepared_df.filter(F.unix_timestamp("event_ts") <= split_epoch)
    test_df = prepared_df.filter(F.unix_timestamp("event_ts") > split_epoch)

    clv_indexer = StringIndexer(
        inputCol="customer_clv_segment",
        outputCol="customer_clv_segment_idx",
        handleInvalid="keep",
    )
    assembler = VectorAssembler(
        inputCols=FEATURE_NUMERIC_COLS + ["customer_clv_segment_idx"],
        outputCol="features",
        handleInvalid="keep",
    )
    gbt = GBTClassifier(
        labelCol="purchased_label",
        featuresCol="features",
        maxIter=max_iter,
        maxDepth=max_depth,
        seed=42,
    )

    pipeline = Pipeline(stages=[clv_indexer, assembler, gbt])
    model = pipeline.fit(train_df)

    holdout_preds = model.transform(test_df)
    evaluator = BinaryClassificationEvaluator(
        labelCol="purchased_label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    roc_auc = evaluator.evaluate(holdout_preds)

    # Counterfactual Rank-1 Position-Bias Debiasing at inference time
    debiased_input_df = test_df.withColumn("displayed_rank_position", F.lit(1))
    debiased_raw_preds = model.transform(debiased_input_df)

    # Mandatory @skill:gcp-spark rule: convert VectorUDT probability via vector_to_array
    export_ready_df = debiased_raw_preds.select(
        "impression_id",
        "search_event_id",
        "item_id",
        "query_tier",
        vector_to_array(F.col("probability")).getItem(1).alias("predicted_conversion_prob"),
        F.col("prediction").cast("int").alias("predicted_label"),
        "event_date",
    )
    return export_ready_df, roc_auc


def log_run_to_geap_experiments(
    project_id: str,
    location: str,
    experiment_name: str,
    run_name: str,
    params: dict[str, str | int | float],
    metrics: dict[str, float],
) -> dict[str, Any]:
    """Track Managed Spark MLlib training runs in GEAP (fka Vertex AI) Experiments."""
    try:
        from google.cloud import aiplatform

        aiplatform.init(project=project_id, location=location, experiment=experiment_name)
        aiplatform.start_run(run=run_name)
        aiplatform.log_params(params)
        aiplatform.log_metrics(metrics)
        aiplatform.end_run()
        return {
            "tracker": "GEAP (fka Vertex AI) Experiments",
            "experiment": experiment_name,
            "run": run_name,
            "status": "LOGGED_TO_CLOUD",
            "params": params,
            "metrics": metrics,
        }
    except Exception:
        return {
            "tracker": "GEAP (fka Vertex AI) Experiments (Local/Offline Telemetry Mirror)",
            "experiment": experiment_name,
            "run": run_name,
            "status": "MIRRORED_LOCALLY",
            "params": params,
            "metrics": metrics,
        }


try:
    impressions_df = (
        spark.read.format("bigquery")
        .option("table", f"{PROJECT_ID}.raw_retail_core.search_impressions")
        .load()
    )
    print(f"[BigQuery Connector] Loaded {PROJECT_ID}.raw_retail_core.search_impressions")
except Exception:
    impressions_df = spark.read.json(f"gs://{PROJECT_ID}-iceberg-lakehouse/dbx_seed_data/search_impressions.jsonl")
    print("[GCS Seed Fallback] Loaded search_impressions.jsonl")

predictions_df, roc_auc = train_and_score_spark_mllib(impressions_df, max_iter=25, max_depth=5)
print(f"Chronological Holdout ROC AUC: {roc_auc:.4f}")
predictions_df.printSchema()

geap_log = log_run_to_geap_experiments(
    project_id=PROJECT_ID,
    location=LOCATION,
    experiment_name="clarity-spark-mllib-reranker-exp",
    run_name="notebook-gbt-rank1-debiased-v1",
    params={
        "engine": "Managed Service for Apache Spark (Serverless Runtime 2.3)",
        "session_template": SESSION_TEMPLATE,
        "algorithm": "GBTClassifier",
        "max_iter": 25,
        "max_depth": 5,
        "debiasing": "counterfactual_rank_1",
    },
    metrics={"holdout_roc_auc": round(float(roc_auc), 4)},
)
print("Logged run to GEAP (fka Vertex AI) Experiments:", geap_log)